# Step 6: Final Solution
## Horizon-Aware Startup Outcome Prediction

## 6.1 Final Model Selection Summary

### Selection Rationale

The model selection follows the project's core methodological principle: **choose from valid horizons, not the highest absolute score.** H3 (full snapshot) results demonstrate leakage inflation but must not drive deployment decisions — those lifetime funding aggregates are unavailable at the time a prediction would be useful.

### Three Final Artefacts

| Role | Model | Horizon | Test AUC | Val AUC | Purpose |
|------|-------|---------|----------|---------|---------|
| **Primary model** | CatBoost | H1 (Founding-Time) | 0.674 | 0.702 | Most defensible — answers the cleanest causal question with zero temporal contamination |
| **Robustness model** | CatBoost | H2 (First-Funding) | 0.700 | 0.711 | Practical compromise — adds time-to-first-funding signal (+0.026 AUC) |
| **Leakage demonstration** | CatBoost | H3 (Full Snapshot) | 0.770 | 0.778 | Upper bound — shows +0.096 AUC inflation from temporally contaminated features |

### Why CatBoost?

CatBoost dominates all three horizons on both validation and test sets. The key advantage is **native categorical handling**: the dataset's high-cardinality fields (468 primary categories, 407 markets) contain signal that frequency encoding (used by LR) and ordinal encoding (used by HGB) cannot fully capture. CatBoost's ordered target encoding captures category-outcome relationships that other encodings lose.

### Why H1 as Primary?

H1 uses only features provably available at founding time (sector, geography, founding date). This is the only horizon where the model's predictions reflect genuine early signal rather than retrospective funding history. The 0.674 test AUC is modest but honest — it represents what you could actually predict about a startup before any funding activity occurs.

### The H1→H2→H3 Progression

- **H1→H2 (+0.026):** First-funding timing adds limited signal beyond founding-time features. The time-to-first-funding lag provides a noisy proxy for investor interest, but does not substantially change predictions.
- **H2→H3 (+0.070):** Lifetime funding aggregates introduce major apparent improvement. This is the leakage gap — these features encode information that accumulated during and after the outcome process. Quantitatively, the H3–H1 gap (+0.096) represents 36% of H3's total discrimination above chance: (0.770 − 0.674) / (0.770 − 0.500) = 0.096 / 0.270 = 0.356. Over a third of H3's apparent predictive power is attributable to temporally contaminated features.

## 6.2 Model Card — CatBoost H1 (Primary Model)

| Field | Detail |
|-------|--------|
| **Model name** | CatBoost Classifier — Founding-Time Horizon (H1) |
| **Version** | 1.0 (trained on Crunchbase terminal-outcome subset, train split ≤2008) |
| **Intended use** | Early-stage probabilistic assessment of startup terminal outcome (acquired vs closed) using only features observable at founding time. Suitable for portfolio-level screening, cohort analysis, and research into founding-time predictors of startup outcomes. |
| **Not intended for** | Real-time investment decisions on individual startups; startup valuation or pricing; startups outside Crunchbase coverage; startups founded after 2013 (dataset vintage); jurisdictions not represented in training data; as a sole decision-making tool without human judgement. |
| **Target definition** | Binary: acquired = 1, closed = 0. All 41,829 operating startups excluded as right-censored observations — they have not reached a terminal outcome, and treating them as a class introduces survivorship bias. |
| **Horizon definition** | H1 (Founding-Time): 11 features provably available at the time of founding — sector/category, market, geography (country, state), founding year/quarter, category count, USA flag, state presence. No funding amounts, funding dates, or funding round information. |
| **Excluded populations** | (1) Operating startups (right-censored, n=41,829); (2) Startups without valid status label (n=1,314); (3) 4,856 blank rows and 4 duplicate rows removed during cleaning (n=4,860 total). |
| **Training data** | 3,218 startups with first funding year ≤2008 (65.6% acquired, 34.4% closed). |
| **Validation data** | 1,625 startups, first funding years 2009–2010 (50.3% acquired, 49.7% closed). |
| **Test data** | 1,452 startups, first funding year ≥2011 (52.6% acquired, 47.4% closed). |
| **Evaluation protocol** | Temporal train/val/test split (no random shuffling). Primary metric: ROC-AUC (threshold-independent). Secondary: PR-AUC, balanced accuracy, F1, Brier score, ECE. Test set opened once after all tuning. |
| **Performance (test)** | ROC-AUC = 0.674, PR-AUC = 0.689, Balanced Accuracy = 0.615, F1 = 0.525, Brier = 0.239, ECE = 0.114 |
| **Calibration status** | Uncalibrated ECE = 0.114. Platt scaling (fitted on validation set) reduces ECE to 0.047 (59% reduction) and Brier score from 0.239 to 0.228. **Platt-calibrated probabilities recommended for deployment.** |
| **Hyperparameters** | Tuned via Optuna (30 trials) with validation-based early stopping. Native categorical handling for primary_category, market_clean, country_code, state_code, region. |
| **Major risks and caveats** | (1) **Limited feature set:** founding-time features provide modest discrimination (AUC 0.674) — the model is informative but not highly predictive; (2) **Dataset vintage:** Crunchbase snapshot is pre-2015; startup ecosystem dynamics may have shifted; (3) **Selection bias:** only startups that appear in Crunchbase are represented — survivorship bias towards venture-backed and tech-sector startups; (4) **Temporal drift:** 15-point acquired-rate drop from train (65.6%) to val (50.3%) reflects temporal acquisition-rate decay — earlier cohorts had more time to reach acquisition; (5) **Geographic concentration:** training data over-represents USA-based startups; performance on underrepresented geographies (e.g., CAN AUC=0.519) is substantially worse; (6) **founding_year dominance:** SHAP analysis shows founding_year is the top feature — partly reflects temporal acquisition-rate patterns rather than pure causal signal. |

## 6.3 Final Performance Table

### All Models — Test Set Results

| Horizon | Model | ROC-AUC | PR-AUC | Bal. Acc. | F1 | Brier | ECE |
|---------|-------|---------|--------|-----------|-----|-------|-----|
| **H1** | LogisticRegression | 0.665 | 0.683 | 0.561 | 0.349 | 0.264 | 0.184 |
| **H1** | HistGradientBoosting | 0.659 | 0.679 | 0.614 | 0.621 | 0.232 | 0.056 |
| **H1** | **CatBoost** | **0.674** | **0.689** | **0.615** | 0.525 | 0.239 | 0.114 |
| **H1** | TabM | 0.668 | 0.689 | 0.620 | 0.549 | 0.248 | 0.105 |
| **H2** | LogisticRegression | 0.640 | 0.652 | 0.559 | 0.370 | 0.243 | 0.084 |
| **H2** | HistGradientBoosting | 0.671 | 0.680 | 0.615 | 0.667 | 0.227 | 0.031 |
| **H2** | **CatBoost** | **0.700** | **0.708** | **0.621** | 0.534 | 0.230 | 0.096 |
| **H2** | TabM | 0.674 | 0.688 | 0.582 | 0.407 | 0.240 | 0.113 |
| **H3** | LogisticRegression | 0.683 | 0.714 | 0.575 | 0.308 | 0.244 | 0.056 |
| **H3** | **CatBoost** | **0.770** | **0.774** | **0.639** | 0.500 | 0.236 | 0.188 |

*Note: DummyClassifier baselines (AUC=0.500) were evaluated on validation only as a sanity check and are not included in the test-set table. All 10 rows above correspond to the 10 test-set entries in `modelling_results.csv`.*

### Cross-Horizon Leakage Gap (CatBoost, Test Set)

| Comparison | AUC Difference | Interpretation |
|------------|---------------|----------------|
| H3 − H1 | **+0.096** | Total leakage inflation from lifetime funding aggregates |
| H3 − H2 | +0.070 | Gain from adding funding amounts beyond first-funding timing |
| H2 − H1 | +0.026 | Marginal value of first-funding timing information |

## 6.4 Key Findings

### 6.4.1 The Horizon Gap — The Signature Finding

The cross-horizon AUC comparison (Figure 14) is the centrepiece of this project. CatBoost test AUC inflates from **0.674 (H1) → 0.700 (H2) → 0.770 (H3)**, a total gap of +0.096.

This gap demonstrates that lifetime funding aggregates — which are the standard features most Crunchbase-based studies use — introduce substantial temporal contamination. A model trained on H3 features appears to predict startup outcomes well (AUC 0.770), but much of this "predictive power" comes from variables that encode information accumulated *during and after* the outcome process.

The modest H1→H2 gain (+0.026) suggests that knowing *when* a startup first received funding adds limited marginal signal beyond what sector, geography, and founding date already provide. The large H2→H3 gain (+0.070) confirms that the real performance inflation comes from funding amounts and round counts — information that is only fully known retrospectively.

**Quantifying the leakage share:** The H3–H1 gap (+0.096) as a fraction of H3's total discrimination above chance (AUC − 0.5): (0.770 − 0.674) / (0.770 − 0.500) = 0.096 / 0.270 = **35.6%**. Over a third of H3's apparent predictive power is attributable to temporally contaminated features.

**Implication:** Studies that report high predictive accuracy on startup outcomes using lifetime funding features may be overstating the genuinely predictable signal. The honest predictive ceiling using only founding-time information is modest (AUC ~0.67).

### 6.4.2 Feature Importance Insights

SHAP analysis of CatBoost H1 (Figure 19) reveals the following feature importance ranking by mean |SHAP value|:

1. **founding_year** — Dominant feature. Higher founding years push predictions toward "closed." This reflects temporal acquisition-rate decay: startups founded earlier had more elapsed time to reach acquisition. This is a legitimate temporal pattern but not a causal relationship — the model partly learns *when* a startup was founded rather than *what* about it predicts outcomes.

2. **state_code** — Geographic sub-national signal. Certain US states (e.g., California, New York, Massachusetts) are associated with higher acquisition rates, consistent with the concentration of venture capital and acquirers in these regions.

3. **market_clean / primary_category** — Sector-level signal. Some markets and categories (e.g., Software, E-Commerce) have historically higher acquisition rates than others (e.g., Biotechnology, Health Care).

4. **founding_quarter** — Weak seasonal signal. Captures minor timing effects within years.

5. **is_usa / country_code** — US-based startups show different outcome distributions from non-US startups, reflecting the geographic concentration of the venture-backed startup ecosystem.

All models agree on founding_year and state_code as top features (Figure 21). The consistency across model classes (linear, tree-based, deep learning) validates that these features carry genuine signal, though the interpretation of founding_year requires careful temporal reasoning.

### 6.4.3 Model Comparison Insights

**Tree-based models vs linear models:** On H1, CatBoost (0.674) modestly outperforms LogisticRegression (0.665) and HGB (0.659). The small gap suggests limited nonlinear interaction effects in founding-time features — most signal is captured by main effects. CatBoost's edge comes from its superior categorical encoding, not from modelling complex interactions.

**Deep learning (TabM) vs gradient-boosted trees:** TabM achieves H1 test AUC of 0.668 — competitive with but not exceeding CatBoost (0.674). On this dataset (6,295 rows, 11 features at H1), gradient-boosted trees have sufficient capacity to capture the available signal. TabM's batch-ensembled MLP architecture (best H1 configuration: k=8 ensemble members) shows clean convergence (Figure 13) but does not find additional structure. This is consistent with recent tabular ML benchmarks: deep learning methods approach but rarely exceed well-tuned gradient-boosted trees on small-to-medium tabular datasets.

**Calibration matters:** Uncalibrated CatBoost H1 has ECE of 0.114 — predicted probabilities are systematically miscalibrated. Platt scaling reduces ECE to 0.047 (59% improvement) with minimal AUC loss. For any deployment scenario where probability estimates inform decisions, post-hoc calibration is essential.

**Val-to-test degradation:** All models show 1–3 AUC point drops from validation to test. This is expected with chronological splits — the model must generalise to a future time period with different base rates (train: 65.6% acquired, test: 52.6%). The modest degradation confirms the models are not severely overfit to the validation period.

## 6.5 Limitations and Risks

### Data Limitations

1. **Dataset vintage:** The Crunchbase snapshot is pre-2015. The startup ecosystem has evolved substantially — new sectors (AI/ML, crypto, climate tech), new funding patterns (mega-rounds, SPACs), and new geographic centres have emerged. Model predictions may not transfer to contemporary startups.

2. **Selection bias:** Crunchbase over-represents venture-backed, technology-sector, and US-based startups. Startups that never sought or received venture funding, operated in non-tech sectors, or were based outside major startup hubs are underrepresented. The model's predictions are conditioned on this biased sample.

3. **Right-censoring:** The 41,829 operating startups were excluded because they have not reached a terminal outcome. Some of these will eventually be acquired or close. The terminal-outcome subset (n=6,295) may not be representative of all startups — it over-represents those that reached outcomes faster, potentially biasing toward startups with clearer trajectories.

4. **Outcome definition:** "Acquired" encompasses a wide range of outcomes, from successful exits to acqui-hires to distressed acquisitions. The binary target treats all acquisitions equally, masking substantial heterogeneity in outcome quality.

### Methodological Limitations

5. **Feature poverty at H1:** Only 11 features are available at founding time. Key predictors of startup success — team quality, product-market fit, competitive dynamics, founding team track record — are not captured in the Crunchbase schema. The modest AUC (0.674) reflects this fundamental feature limitation.

6. **Temporal confounding:** founding_year is the dominant SHAP feature, but its importance partly reflects temporal acquisition-rate decay (earlier cohorts had more time to exit) rather than a genuine causal relationship between founding year and outcome. This limits the model's interpretive value for understanding *why* some startups succeed.

7. **Geographic heterogeneity:** Performance varies substantially across geographies (GBR AUC=0.723, CAN AUC=0.519). A single global model cannot capture market-specific dynamics, regulatory environments, and ecosystem characteristics that differ across countries.

8. **Sectoral heterogeneity:** Biotechnology has the highest error rate (55.3%). Sectors with long development cycles, regulatory hurdles, or binary technical risk profiles are systematically harder to predict using founding-time features.

### Deployment Risks

9. **Probability miscalibration:** Without Platt scaling, predicted probabilities are unreliable (ECE=0.114). Any use of raw model outputs for decision-making requires post-hoc calibration.

10. **Distributional shift:** The 15-point acquired-rate drop from train to test demonstrates temporal drift. In a real deployment, the model would need periodic recalibration as the underlying base rate of acquisitions evolves.

## 6.6 Recommendations and Next Steps

### For Practitioners

1. **Use Platt-calibrated probabilities.** The raw CatBoost outputs are miscalibrated (ECE=0.114). Post-hoc calibration (ECE=0.047) produces substantially more reliable probability estimates at negligible computational cost.

2. **Treat predictions as screening-level, not decision-level.** An AUC of 0.674 means the model has informative but limited discriminative power. Use it for portfolio-level risk stratification, not for individual investment decisions.

3. **Monitor for distributional shift.** The model was trained on pre-2009 data and tested on 2011+ data. If deployed on contemporary startups, recalibrate regularly as the startup ecosystem evolves.

4. **Interpret founding_year effects carefully.** The model's reliance on founding year reflects temporal base-rate patterns, not a causal claim that "earlier startups are inherently more likely to be acquired."

### For Future Research

5. **Richer feature sets.** Incorporate team-level features (founder experience, education, prior exits), product-level features (technology category, patent filings), and network-level features (co-investor networks, board connections). These are known predictors of startup outcomes that Crunchbase's simplified schema does not capture.

6. **Survival analysis framing.** Instead of binary classification on the terminal-outcome subset, model time-to-event (acquisition or closure) using Cox proportional hazards or parametric survival models. This properly handles right-censored (operating) observations rather than excluding them.

7. **Contemporary data.** Repeat the horizon-aware analysis on a recent Crunchbase snapshot (2020+) to assess whether the leakage gap persists and whether founding-time feature importance has shifted.

8. **Region-specific models.** The large geographic performance variation (GBR AUC=0.723 vs CAN AUC=0.519) suggests that region-specific models, or models with region-level random effects, could improve predictions for underrepresented geographies.

9. **Multi-class or ordinal outcome.** Distinguish between successful acquisitions, acqui-hires, and distressed acquisitions (if data allows) to model outcome quality rather than just outcome occurrence.

## 6.7 Agent Reflection

### How the AI Agent Was Used

The AI agent (Claude Code) served as a code generation and analysis assistant throughout all six phases of this project. Its contributions ranged from project scaffolding and data cleaning to model training and evaluation visualisation. Every agent output was subject to human verification before acceptance.

### Key Agent Contributions

The Decision Register (`logs/DECISION_REGISTER.md`) documents 61 agent interactions across 6 phases. Summary:

- **Accepted (36):** Scaffolding, metric implementations, standard model training code, figure generation, evaluation analyses
- **Accepted-Modified (11):** Cases where agent output was functionally correct but needed refinement — e.g., improved notebook structure, better emphasis on leakage risks in comments, permutation importance replacing unavailable feature_importances_
- **Rejected (14):** Substantive analytical errors the agent made and the user caught

### Agent Mistakes Caught and Corrected

1. **Funding feature leakage (ID 1):** Agent initially proposed using all funding columns as standard features, treating lifetime aggregates as point-in-time data. This would have introduced temporal contamination into every model. User rejected and designed the three-horizon framework. *This single correction defined the entire project's methodology.*

2. **Operating class censoring (ID 4):** Agent did not flag that the 41,829 operating startups are right-censored observations. Including them as a third class or as implicit failures would have contaminated the target variable with survivorship bias. User identified and excluded them.

3. **Random split proposed (ID 3):** Agent proposed random stratified 70/15/15 splits instead of temporal validation. User replaced with chronological split by first_funding_year, which better reflects real-world deployment conditions.

4. **Incorrect numbers in logs (ID 22):** Agent produced incorrect counts in EDA summaries — 1 duplicate pair (actual: 2), 2,739 impossible dates (actual: 2,745), 6,170 missing status (actual: 1,314). User cross-checked all numbers against notebook outputs.

5. **Distribution shift understatement (ID 32):** Agent described a 15-point class balance drift (train 65.6% vs val 50.3%) as "sensible" without flagging the magnitude. User identified this as a significant evaluation risk requiring discussion.

6. **Six factual errors in Phase 6 outputs (ID 61):** Agent fabricated a Dummy test row not in source CSV, miscounted register entries, confused H1/H2 TabM hyperparameters, propagated stale feature counts, conflated count compositions, and cited an unverifiable "~60%" with no formula (actual: 36%). User caught all six by cross-checking against source data.

### Reflection on Agent Governance

The agent excels at code generation speed, boilerplate production, and parallel exploration of model architectures. It struggles with *temporal reasoning* (when information becomes available), *statistical interpretation* (significance of distribution shifts), and *number accuracy* (propagating computed values to prose). The governance model — generate, verify, accept/modify/reject — caught 14 substantive errors that would have compromised the project's analytical integrity. The most consequential agent failure (funding leakage) became the project's most valuable methodological contribution.

See `logs/DECISION_REGISTER.md` for the complete interaction log.